# Fitur beku ResNet18 + Logistic Regression
        **vision › classification** · notebook `transfer_resnet18_logreg`

        **Kapan dipakai:** Transfer learning tercepat: ekstrak fitur ResNet18 (bobot ImageNet beku) sekali, latih classifier linear. Kuat dengan data berlabel sedikit, tanpa fine-tuning.

        **Jebakan umum:** Fitur ImageNet kurang cocok untuk domain sangat berbeda (medis, satelit): fine-tuning lebih baik. Gambar kecil di-upsample menurunkan kualitas fitur.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** CIFAR-10 subset (4000 latih / 1000 uji, 10 kelas, 32x32). Bobot pretrained dibaca dari `pretrained/` (offline); GPU dipakai otomatis bila ada.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba. Format data: folder-per-kelas atau CSV `path,label`.

In [ ]:
DATA_ROOT  = ROOT / "data_latihan/vision/classification/cifar10_small"            # folder gambar
TRAIN_CSV  = DATA_ROOT / "train.csv"   # kolom: path (relatif ke DATA_ROOT), label
TEST_CSV   = DATA_ROOT / "test.csv"    # kolom: id, path (tanpa label)
TEST_LABELS = DATA_ROOT / "test_labels.csv"  # kunci jawaban test (hanya untuk latihan); None di lomba
SEED, VALID_SIZE = 42, 0.2
MODEL_KEY = "transfer_resnet18_logreg"
OUT = out_dir("vision", "classification", MODEL_KEY)
set_seed(SEED)
BACKBONE = "resnet18"; SIZE = 96

## 2. Baca gambar

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from superdb.vision import load_images, show_grid
tr = pd.read_csv(TRAIN_CSV); te = pd.read_csv(TEST_CSV)
le = LabelEncoder(); y = le.fit_transform(tr["label"]); CLASSES = list(le.classes_)
X = load_images(tr["path"], DATA_ROOT); Xte = load_images(te["path"], DATA_ROOT)
print(X.shape, Xte.shape, "| kelas:", CLASSES); show_grid(X[::400], [CLASSES[i] for i in y[::400]], n=10); plt.show()
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)

## 3. Ekstraksi fitur dari backbone beku (sekali, di-cache)
Bobot beku => tidak ada yang dipelajari dari data, jadi aman dihitung sebelum split.

In [ ]:
from pathlib import Path
from superdb.vision import extract_features
import time; t0 = time.time()
F, Fte = extract_features(BACKBONE, X, SIZE), extract_features(BACKBONE, Xte, SIZE)
print(f"fitur {F.shape} dalam {time.time() - t0:.0f} dtk")
Ftr, Fva, ytr_, yva_ = train_test_split(F, y, test_size=VALID_SIZE, random_state=SEED, stratify=y)

## 4. Classifier linear di atas fitur

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000))

## 5. CV, evaluasi, dan perbandingan dengan baseline HOG

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = cross_val_score(model, Ftr, ytr_, cv=StratifiedKFold(3, shuffle=True, random_state=SEED), scoring="f1_macro"); print(f"CV f1_macro = {cv.mean():.4f} ± {cv.std():.4f}")
model.fit(Ftr, ytr_); pred = model.predict(Fva); proba = model.predict_proba(Fva)
metrics = classification_metrics(yva_, pred, proba, classes=np.arange(len(CLASSES))); display(show_table(metrics))
plot_confusion(yva_, pred, labels=np.arange(len(CLASSES)), path=OUT / "confusion.png"); plt.show()

## 6. Latih ulang pada semua data → submission → simpan

In [ ]:
from sklearn.base import clone
final = clone(model).fit(F, y); CLASSES_PRED = final.predict(Fte)

In [ ]:
pred_te = CLASSES_PRED
sub = pd.DataFrame({"id": te["id"], "label": [CLASSES[i] for i in pred_te]}); sub.to_csv(OUT / "submission.csv", index=False); display(sub.head(3))
if TEST_LABELS is not None and Path(TEST_LABELS).exists():
    truth = pd.read_csv(TEST_LABELS); acc = float((truth.set_index("id").loc[sub["id"], "label"].to_numpy() == sub["label"].to_numpy()).mean())
    print(f"akurasi pada test (kunci jawaban): {acc:.4f}"); metrics["test_accuracy"] = acc

In [ ]:
save_model({"clf": final, "classes": CLASSES, "backbone": BACKBONE, "size": SIZE}, OUT / "model.joblib"); save_json({"cv_f1_macro": float(cv.mean()), "valid": metrics, "feature_dim": int(F.shape[1])}, OUT / "metrics.json")
b = load_model(OUT / "model.joblib"); assert (b["clf"].predict(Fte[:30]) == final.predict(Fte[:30])).all(); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan backbone: ViT-B/16 & ConvNeXt biasanya terbaik, MobileNet tercepat.
- Fitur beku kurang cocok untuk domain jauh dari ImageNet: gunakan `finetune_*`.
- Gabungkan fitur beberapa backbone (hstack) lalu classifier.